In [1]:
import os
print(os.listdir("."))

['.config', 'sample_data']


In [2]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
!pip install transformers datasets accelerate scikit-learn

In [ ]:
import os, torch, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)

if not os.path.exists("all-data.csv"):
    from google.colab import files
    files.upload()

df = pd.read_csv("all-data.csv", encoding="latin-1", names=["sentiment", "text"])


X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)


name = "ProsusAI/finbert"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)
label2id = {k.lower(): v for k, v in model.config.label2id.items()}
id2label = {v: k for k, v in label2id.items()}


class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(texts.tolist(), truncation=True, padding=True, max_length=128)
        self.labels = [label2id[l] for l in labels]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds = DS(X_train, y_train)
test_ds = DS(X_test, y_test)

args = TrainingArguments(
    output_dir="finbert-tuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

out = trainer.predict(test_ds)
pred_ft = [id2label[i] for i in out.predictions.argmax(-1)]
print("Accuracy:", (pd.Series(pred_ft, index=y_test.index) == y_test).mean())
print(classification_report(y_test, pred_ft))

In [ ]:
import os
os.rename("all-data (1).csv", "all-data.csv")
print(os.listdir("."))

In [ ]:
trainer.save_model("my_finbert")
tok.save_pretrained("my_finbert")
!zip -r my_finbert.zip my_finbert
from google.colab import files
files.download("my_finbert.zip")
